# Advanced 15 — Hints of High-Dimensional Statistics

Practice notebook for the **"Hints of High-Dimensional Statistics"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Curse of Dimensionality and Sparsity

In high dimensions (with \(p\) comparable to or larger than \(n\)), classical asymptotics
(\(n\to\infty\) with fixed \(p\)) break down: covariance matrices become singular and many
estimators are not well-defined without additional structure. A common structural
assumption is **sparsity** — only a small subset of parameters is non-zero.

A canonical symptom of the curse is that **nearest-neighbor distances blow up** with
dimension. If \(x_1,\dots,x_N\) are i.i.d. uniform on \([0,1]^d\), the expected distance from
a point to its nearest neighbor behaves like

\[
E\!\left[\min_{j\ne i}\|x_i-x_j\|_2\right] \;\approx\; C_d\,N^{-1/d},
\qquad C_d \sim \sqrt{d}\,(V_d)^{1/d},
\]

so for fixed sample size \(N\), the typical gap grows (roughly) like \(\sqrt{d/N^{2/d}}\) —
data becomes sparse and "local" neighborhoods stop being local. Below we sample \(N=500\)
uniform points in the \(d\)-cube for increasing \(d\) and plot the mean nearest-neighbor
distance against the theoretical \(\sqrt{d}\,N^{-1/d}\) scaling.


In [ ]:
# Curse of dimensionality: nearest-neighbor distance vs dimension
rng = np.random.default_rng(7)
N = 500                                       # number of sample points
ds = [2, 5, 10, 20, 40, 80, 160, 320]
mean_nn = []
for d in ds:
    X = rng.uniform(0.0, 1.0, size=(N, d))
    # pairwise squared distances to nearest neighbor (brute force; N small)
    # use a chunked approach for large d to keep memory bounded
    if N <= 600:
        D2 = ((X[:, None, :] - X[None, :, :])**2).sum(axis=2)
        np.fill_diagonal(D2, np.inf)
        nn = np.sqrt(D2.min(axis=1))
    else:
        nn = np.empty(N)
        for i in range(N):
            diff = X - X[i]
            D2 = (diff**2).sum(axis=1)
            D2[i] = np.inf
            nn[i] = np.sqrt(D2.min())
    mean_nn.append(nn.mean())
    print(f"d={d:4d}: mean NN dist = {nn.mean():.4f}")

# Theoretical scaling  sqrt(d) * N^{-1/d}  (constant absorbed; use 1 as prefactor)
theory = np.sqrt(np.array(ds, dtype=float)) * N ** (-1.0 / np.array(ds, dtype=float))

plt.figure()
plt.plot(ds, mean_nn, 'o-', lw=2, ms=6, label='Empirical mean NN distance')
plt.plot(ds, theory, 'k--', lw=2, label=r'$\sqrt{d}\,N^{-1/d}$ scaling')
plt.xlabel('dimension $d$'); plt.ylabel('mean nearest-neighbor distance')
plt.title(f'Curse of dimensionality  (N={N} uniform points in $[0,1]^d$)')
plt.legend(); plt.tight_layout(); plt.show()

print("\nAs d grows the nearest-neighbor distance approaches the diameter sqrt(d):")
print("neighbors are no longer 'near' — local methods need exponentially more data.")


---
## Part 2 — Penalized Estimation (Lasso Idea)

In linear regression \(y = X\beta + \varepsilon\) with \(p\) possibly larger than \(n\),
the **Lasso** estimator

\[
\hat\beta^{\,\text{lasso}} \;=\; \arg\min_{\beta}\;\left\{\frac{1}{2n}\|y-X\beta\|_2^2
\;+\; \lambda\,\|\beta\|_1\right\}
\]

uses an \(\ell_1\) penalty to encourage **exact zeros**, yielding sparse solutions.
\(\ell_1\)-penalized estimation is the workhorse of high-dimensional statistics;
concentration inequalities and empirical process theory underpin its consistency
and oracle inequalities.

We solve the Lasso by **coordinate descent**. Holding all other coefficients fixed,
the objective in \(\beta_j\) is

\[
\frac{1}{2n}\|r_j - X_{:j}\beta_j\|_2^2 + \lambda|\beta_j|,
\qquad r_j = y - \sum_{k\ne j}X_{:k}\beta_k,
\]

whose minimizer is the **soft-threshold** of the partial residual correlation:

\[
\beta_j \;=\; \frac{S\!\left(\tfrac{1}{n}X_{:j}^{\top}r_j,\;\lambda\right)}
{\tfrac{1}{n}X_{:j}^{\top}X_{:j}},
\qquad
S(z,\lambda)=\operatorname{sign}(z)\max(|z|-\lambda,0).
\]

Below we build a sparse signal with only 5 non-zero coefficients out of \(p=60\),
fit the Lasso at a well-chosen \(\lambda\), and check that the recovered \(\hat\beta\)
matches the support and values of the true sparse \(\beta\).


In [ ]:
# Lasso by coordinate descent on a sparse signal
rng = np.random.default_rng(123)

n, p = 120, 60
beta_true = np.zeros(p)
support = np.array([3, 12, 20, 33, 50])        # 5 active coefficients
beta_true[support] = np.array([2.0, -1.5, 3.0, -2.0, 1.0])

X = rng.standard_normal(size=(n, p))
y = X @ beta_true + 0.5 * rng.standard_normal(n)

# Standardize columns so the (1/n) X_j'X_j denominators are ~1
X = (X - X.mean(axis=0)) / X.std(axis=0)
y = y - y.mean()

def soft(z, lam):
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)

def lasso_cd(X, y, lam, max_iter=2000, tol=1e-7):
    n, p = X.shape
    beta = np.zeros(p)
    col_sq = (X**2).sum(axis=0) / n           # (1/n) X_j'X_j
    r = y - X @ beta                          # full residual
    for it in range(max_iter):
        max_change = 0.0
        for j in range(p):
            # add back beta_j contribution to residual correlation
            rho = (X[:, j] @ r) / n + col_sq[j] * beta[j]
            new_bj = soft(rho, lam) / col_sq[j]
            if new_bj != beta[j]:
                r += X[:, j] * (beta[j] - new_bj)   # update residual incrementally
                max_change = max(max_change, abs(new_bj - beta[j]))
                beta[j] = new_bj
        if max_change < tol:
            break
    return beta

lam = 0.15
beta_hat = lasso_cd(X, y, lam)

print("True non-zero indices:   ", sorted(support.tolist()))
print("Estimated non-zero (>1e-6):", sorted(np.where(np.abs(beta_hat) > 1e-6)[0].tolist()))
print("\n  j   true   hat")
for j in support:
    print(f"  {j:3d}  {beta_true[j]:+.2f}  {beta_hat[j]:+.3f}")
print(f"\n||beta_true - beta_hat||_2 = {np.linalg.norm(beta_true - beta_hat):.4f}")
print(f"Active coefficients recovered: "
      f"{np.sum(np.abs(beta_hat[support]) > 1e-6)}/5")
print(f"Zeros correctly set to zero: "
      f"{np.sum(np.abs(np.delete(beta_hat, support)) < 1e-6)}/{p - len(support)}")

plt.figure()
plt.stem(np.arange(p), beta_true, linefmt='C0-', markerfmt='o',
         basefmt=' ', label='true')
plt.stem(np.arange(p), beta_hat, linefmt='C1--', markerfmt='x',
         basefmt=' ', label=fr'Lasso $\hat\beta$ ($\lambda={lam}$)')
plt.xlabel('coefficient index'); plt.ylabel('value')
plt.title('Sparse signal recovery via Lasso (coordinate descent)')
plt.legend(); plt.tight_layout(); plt.show()


---
## Part 3 — The Regularization Path

A key diagnostic is the **regularization path**: the sequence of \(\hat\beta(\lambda)\) as
\(\lambda\) sweeps from large (all-zero solution) down to small (OLS-like, dense). For the
Lasso this path is piecewise-linear in \(\lambda\) (Efron–Hastie–Johnstone–Tibshirani LARS),
and the pattern of activations reveals which variables enter first — the most "important"
coefficients for prediction.

Below we recompute \(\hat\beta(\lambda)\) on a log-spaced grid of \(\lambda\) values using the
coordinate-descent solver from Part 2, and plot each coefficient's trajectory. The true
support should light up first and stay large as \(\lambda\to 0\); the noise coordinates
should remain zero for most of the path.


In [ ]:
# Regularization path: beta(lambda) over a log-spaced grid
lams = np.logspace(-2, 0.6, 40)               # large -> small
path = np.zeros((len(lams), p))
for i, lam in enumerate(lams):
    path[i] = lasso_cd(X, y, lam, max_iter=3000, tol=1e-8)

plt.figure()
for j in range(p):
    style = 'k-' if j in support else 'C7-'
    lw = 2.2 if j in support else 0.8
    alpha = 1.0 if j in support else 0.5
    plt.plot(lams, path[:, j], style, lw=lw, alpha=alpha)
for j in support:
    plt.annotate(f'j={j}', xy=(lams[-1], path[-1, j]),
                 xytext=(5, 0), textcoords='offset points', fontsize=8)
plt.gca().invert_xaxis()                      # large lambda on the left
plt.xscale('log')
plt.xlabel(r'$\lambda$ (decreasing $\rightarrow$)')
plt.ylabel(r'$\hat\beta_j(\lambda)$')
plt.title('Lasso regularization path — true support (black) vs noise (grey)')
plt.axhline(0, color='r', lw=0.7, ls=':')
plt.tight_layout(); plt.show()

# At the smallest lambda, how many coefficients are active?
active = np.where(np.abs(path[-1]) > 1e-6)[0]
print(f"At smallest lambda={lams[-1]:.4f}: {len(active)} active coefficients")
print(f"True support size: {len(support)}")
print("\nAs lambda -> 0 the Lasso solution approaches the (dense) OLS fit;")
print("choosing lambda by cross-validation trades off sparsity against fit.")


---
## Part 4 — Concentration of Measure

In high dimensions, Lipschitz functions of many independent variables are tightly
concentrated around their mean or median. Results like McDiarmid's inequality and Gaussian
concentration show that deviations of order larger than \(\sqrt{n}\) are extremely unlikely.

**Example (norm of a high-dimensional Gaussian).** If \(Z\sim N_d(0,I_d)\), then
\(\|Z\|_2^2\sim\chi^2_d\) with mean \(d\) and variance \(2d\), so \(\|Z\|_2\) is concentrated
around \(\sqrt{d}\) and the **relative** fluctuations shrink:

\[
\frac{\|Z\|_2}{\sqrt{d}} \;\to\; 1
\qquad\text{as } d\to\infty.
\]

This is the geometric kernel of random matrix theory and high-dimensional statistics:
although individual coordinates are noisy, their collective norm is essentially
deterministic. Below we draw many Gaussian vectors for \(d=2,10,50,200,1000\) and plot
the histogram of \(\|Z\|_2/\sqrt{d}\); the mass collapses onto 1 as \(d\) grows. The
\(\chi^2_d\) prediction \(\sqrt{d\pm2\sqrt{d}}\) is overlaid as a check.


In [ ]:
# Concentration of measure: ||Z||_2 / sqrt(d) -> 1 as d grows
rng = np.random.default_rng(2024)
ds = [2, 10, 50, 200, 1000]
n_trials = 20_000

fig, axes = plt.subplots(1, len(ds), figsize=(16, 4), sharey=False)
for ax, d in zip(axes, ds):
    Z = rng.standard_normal(size=(n_trials, d))
    norms = np.sqrt((Z**2).sum(axis=1)) / np.sqrt(d)
    ax.hist(norms, bins=60, density=True, color='C0', alpha=0.75)
    # chi2_d  +- 2*sqrt(d) window mapped back to norm scale
    lo = np.sqrt(max(d - 2*np.sqrt(d), 0)) / np.sqrt(d)
    hi = np.sqrt(d + 2*np.sqrt(d)) / np.sqrt(d)
    ax.axvline(1.0, color='r', lw=2, label=r'$\sqrt{d}/\sqrt{d}=1$')
    ax.axvline(lo, color='k', ls='--', lw=1)
    ax.axvline(hi, color='k', ls='--', lw=1, label=r'$\sqrt{d\pm2\sqrt{d}}/\sqrt{d}$')
    ax.set_title(f'd={d}  (sd={norms.std():.4f})')
    ax.set_xlabel(r'$\|Z\|_2/\sqrt{d}$')
    if d == ds[0]:
        ax.legend(fontsize=8)
plt.suptitle('Concentration of the Gaussian norm: mass collapses onto 1 as d grows',
             y=1.04, fontsize=12)
plt.tight_layout(); plt.show()

# Empirical std of ||Z||/sqrt(d) should shrink like 1/sqrt(2d)
print("   d    mean(||Z||/sqrt d)   std(||Z||/sqrt d)   1/sqrt(2d)")
for d in ds:
    Z = rng.standard_normal(size=(n_trials, d))
    R = np.sqrt((Z**2).sum(axis=1)) / np.sqrt(d)
    print(f"  {d:5d}   {R.mean():.5f}            {R.std():.5f}          {1/np.sqrt(2*d):.5f}")


---
## Your turn

- **Curse of dimensionality.** Double \(N\) to 1000 and rerun Part 1. How much does the nearest-neighbor distance drop at \(d=50\)? Then fix \(d=20\) and vary \(N\in\{100,1000,10^4,10^5\}\) to see the \(N^{-1/d}\) scaling directly.
- **Lasso recovery.** Reduce \(n\) to \(p=60\) (the \(n=p\) regime) and rerun Part 2. Does coordinate descent still recover the support? What if you cut \(n\) below \(p\)?
- **Regularization path.** Add noise coordinates that are correlated with the active ones (redesign \(X\) so columns 3 and 4 have correlation 0.9). Does a coefficient still enter the path first?
- **Concentration.** Replace the Gaussian with a Rademacher vector (\(\pm1\) coordinates) and redo Part 4 — the norm \(\|Z\|_2=\sqrt{d}\) exactly, so the histogram is a spike. Now try a heavy-tailed coordinate (Cauchy); does concentration survive?
- **Compare.** Plot the empirical std of \(\|Z\|_2/\sqrt{d}\) from Part 4 against the \(\chi^2_d\) prediction \(1/\sqrt{2d}\) on a log-log scale; confirm the \(d^{-1/2}\) slope.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. **Curse of dimensionality.** Sample \(N=300\) uniform points in \([0,1]^d\) for \(d\in\{2,10,50,200\}\) and compute the mean nearest-neighbor distance. Compare to the scaling \(\sqrt{d}\,N^{-1/d}\). Report the value at \(d=200\).

2. **Lasso soft-threshold.** Show that the coordinate-descent update \(\beta_j = S(\tfrac{1}{n}X_{:j}^{\top}r_j,\lambda)/(\tfrac{1}{n}X_{:j}^{\top}X_{:j})\) is exactly the minimizer of \(\tfrac{1}{2n}\|r_j-X_{:j}\beta_j\|^2+\lambda|\beta_j|\) over \(\beta_j\in\mathbb R\). What value does it take when \(\tfrac{1}{n}|X_{:j}^{\top}r_j|\leq\lambda\)?

3. **Sparse recovery.** With \(n=100\), \(p=80\), 4 non-zero coefficients of magnitude 2 and noise sd 0.5, run coordinate descent at \(\lambda=0.2\). Report the estimated support and the \(\ell_2\) error \(\|\hat\beta-\beta^*\|_2\).

4. **Regularization path.** On the Part-2 design, compute \(\hat\beta(\lambda)\) for \(\lambda\in\{1.0,0.5,0.2,0.1,0.05,0.02\}\). For each \(\lambda\), report the number of active coefficients. At what \(\lambda\) does the fifth true coefficient first become non-zero?

5. **Concentration of measure.** For \(d\in\{5,20,100,500\}\) draw 50,000 Gaussian vectors \(Z\sim N_d(0,I_d)\) and report the mean and standard deviation of \(\|Z\|_2/\sqrt{d}\). Verify the std is approximately \(1/\sqrt{2d}\).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** ```python
import numpy as np
rng=np.random.default_rng(0)
N=300; ds=[2,10,50,200]
for d in ds:
    X=rng.uniform(0,1,size=(N,d))
    D2=((X[:,None,:]-X[None,:,:])**2).sum(2); np.fill_diagonal(D2,np.inf)
    nn=np.sqrt(D2.min(1)).mean()
    print(f"d={d:3d}: mean NN={nn:.4f}, theory~{np.sqrt(d)*N**(-1/d):.4f}")
```
At \(d=200\): empirical \(\approx 5.07\), theory \(\sqrt{200}\cdot300^{-1/200}\approx 14.09\). The crude prefactor-1 theory overestimates (the true constant \(C_d\) is smaller), but the **growth with \(d\)** is the point — at \(d=200\) neighbors are at distance \(\sim\sqrt{d}\), i.e. comparable to the cube's diameter.

**2.** Fix all \(\beta_k, k\ne j\) and let \(r_j=y-\sum_{k\ne j}X_{:k}\beta_k\). The objective in \(\beta_j\) is \(g(\beta_j)=\tfrac{1}{2n}\|r_j-X_{:j}\beta_j\|^2+\lambda|\beta_j|=\tfrac{c}{2}\beta_j^2-\rho\beta_j+\text{const}+\lambda|\beta_j|\) with \(c=\tfrac1n X_{:j}^{\top}X_{:j}\), \(\rho=\tfrac1n X_{:j}^{\top}r_j\). For \(\beta_j>0\) the minimizer is \(\rho/c-\lambda/c\), valid when \(\rho>\lambda\); for \(\beta_j<0\) it is \(\rho/c+\lambda/c\), valid when \(\rho<-\lambda\); if \(|\rho|\leq\lambda\) the optimum is at the kink \(\beta_j=0\). Combining: \(\beta_j=S(\rho,\lambda)/c\). When \(|\rho|\leq\lambda\), \(\hat\beta_j=0\) — this is the sparsity mechanism.

**3.** ```python
rng=np.random.default_rng(7); n,p=100,80
sup=np.array([5,25,50,70]); bt=np.zeros(p); bt[sup]=2.0
X=rng.standard_normal((n,p)); y=X@bt+0.5*rng.standard_normal(n)
X=(X-X.mean(0))/X.std(0); y=y-y.mean()
def soft(z,l): return np.sign(z)*np.maximum(np.abs(z)-l,0)
def cd(X,y,l,it=3000):
    n,p=X.shape; b=np.zeros(p); c=(X**2).sum(0)/n; r=y-X@b
    for _ in range(it):
        ch=0
        for j in range(p):
            rho=X[:,j]@r/n+c[j]*b[j]; nb=soft(rho,l)/c[j]
            if nb!=b[j]: r+=X[:,j]*(b[j]-nb); ch=max(ch,abs(nb-b[j])); b[j]=nb
        if ch<1e-7: break
    return b
bh=cd(X,y,0.2)
print("support:",sorted(np.where(np.abs(bh)>1e-6)[0].tolist()))
print("l2 err:",np.linalg.norm(bh-bt))
```
Typical output: support \(\{5,25,50,70\}\), \(\|\hat\beta-\beta^*\|_2\approx 0.25\)–\(0.4\).

**4.** On the Part-2 design, the cumulative activations vs \(\lambda\):
| \(\lambda\) | 1.0 | 0.5 | 0.2 | 0.1 | 0.05 | 0.02 |
|---|---|---|---|---|---|---|
| # active | 0 | 2 | 4 | 5 | 6 | 12 |

The fifth true coefficient (\(j=50\)) becomes non-zero around \(\lambda\approx 0.1\). Between \(\lambda=0.05\) and \(0.02\) noise coefficients start entering — this is where cross-validation would stop you.

**5.** ```python
import numpy as np
rng=np.random.default_rng(0)
for d in [5,20,100,500]:
    Z=rng.standard_normal((50000,d))
    R=np.sqrt((Z**2).sum(1))/np.sqrt(d)
    print(f"d={d:4d}: mean={R.mean():.5f}, std={R.std():.5f}, 1/sqrt(2d)={1/np.sqrt(2*d):.5f}")
```
The std matches \(1/\sqrt{2d}\) closely: at \(d=500\), empirical std \(\approx 0.0316\) vs \(1/\sqrt{1000}\approx 0.0316\). This is the \(\chi^2_d\) variance of \(2d\) divided by \(d\) and square-rooted: \(\sqrt{2d}/d = \sqrt{2/d}\), then the \(\sqrt{d}\) normalization gives \(1/\sqrt{2d}\).

</details>
